In [15]:
import pandas as pd
from pathlib import Path

file_name = "C:\\Users\\nnish\\OneDrive\\Desktop\\bank_customer_churn_prediction_dataset.xls"
data_path = Path(file_name)
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
df.head()

Dataset loaded successfully!


,Customer_ID,Age,Account_Type,Account_Balance,Monthly_Transactions,Number_of_Products,Mobile_App_User,Customer_Service_Calls,Months_Inactive,Credit_Card_Holder,Satisfaction_Score,Branch_Visits,Customer_Churned
0,1,26.0,Basic,155808.0,107,5,No,2,15,No,5.0,3,No
1,2,71.0,Basic,186299.0,109,4,Yes,4,10,Yes,2.0,7,No
2,3,32.0,Basic,102173.0,71,5,Yes,0,7,Yes,3.0,7,No
3,4,52.0,Premium,194319.0,19,5,Yes,6,12,Yes,2.0,0,Yes
4,5,28.0,Premium,127968.0,12,3,No,11,15,Yes,2.0,18,No


In [16]:
df = df.drop_duplicates()
df = df.drop(columns=['Customer_ID'])

numeric_columns = ["Age", "Account_Balance", "Satisfaction_Score"]
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

categorical_columns = ["Account_Type", "Mobile_App_User", "Credit_Card_Holder"]
for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values:", df.isnull().sum().sum())
print("Shape:", df.shape)

Missing values: 0
Shape: (380, 12)


In [17]:
target_column = "Customer_Churned"

feature_columns = [
    "Age", "Account_Type", "Account_Balance",
    "Monthly_Transactions", "Number_of_Products",
    "Mobile_App_User", "Customer_Service_Calls",
    "Months_Inactive", "Credit_Card_Holder",
    "Satisfaction_Score", "Branch_Visits"
]

X = df[feature_columns]
y = df[target_column].map({"No": 0, "Yes": 1})

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print(y.value_counts())

Shape of X: (380, 11)
Shape of y: (380,)
Customer_Churned
0    339
1     41
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_features = [
    "Age", "Account_Balance", "Monthly_Transactions",
    "Number_of_Products", "Customer_Service_Calls",
    "Months_Inactive", "Satisfaction_Score", "Branch_Visits"
]

categorical_features = [
    "Account_Type", "Mobile_App_User", "Credit_Card_Holder"
]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed  = preprocessor.transform(X_test)

print("Preprocessing done!")

Preprocessing done!


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_preprocessed, y_train)
print("Logistic Regression trained!")

svm_model = SVC(random_state=42)
svm_model.fit(X_train_preprocessed, y_train)
print("SVM trained!")

Logistic Regression trained!
SVM trained!


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_pred = log_model.predict(X_test_preprocessed)
svm_pred = svm_model.predict(X_test_preprocessed)

print("=== Logistic Regression ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, log_pred))
print("Accuracy:", accuracy_score(y_test, log_pred))
print("Precision:", precision_score(y_test, log_pred))
print("Recall:", recall_score(y_test, log_pred))
print("F1 Score:", f1_score(y_test, log_pred))

print("\n=== SVM ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred))
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))

=== Logistic Regression ===
Confusion Matrix:
[[67  1]
 [ 6  2]]
Accuracy: 0.9078947368421053
Precision: 0.6666666666666666
Recall: 0.25
F1 Score: 0.36363636363636365

=== SVM ===
Confusion Matrix:
[[68  0]
 [ 7  1]]
Accuracy: 0.9078947368421053
Precision: 1.0
Recall: 0.125
F1 Score: 0.2222222222222222


In [24]:
print("""
Summary

Which model performed better?
Looking at the results, both models did a decent job but in different ways.  Logistic Regression was better at catching customers who actually left the 
bank. SVM was more careful and precise but missed some churners. So it  depends on what matters more to the bank.

Which metric is most important for this business problem?
For a bank trying to stop customers from leaving, Recall is the most  important metric. We would rather flag too many customers than miss someone 
who is about to leave. Losing a customer completely is much more costly than sending an extra retention offer to someone who was going to stay anyway.

What do false positives and false negatives mean?
A false positive is when the model thinks a customer will leave but they actually stay. This means the bank spends money on a retention offer they 
did not need. Not a big deal but a waste of resources. A false negative is when the model thinks a customer will stay but they actually leave. 
This is the dangerous one because the bank does nothing and loses the customer completely.

What is one possible limitation or bias?
The biggest problem with this dataset is that only 42 out of 380 customers actually churned. This means the model sees mostly No Churn examples during 
training and gets used to predicting No Churn. It is like studying only passing grades and then trying to predict who will fail.

Why should human judgment still be used?
A model looks at numbers but it cannot understand the full picture. For example a customer might have low transactions because they were sick or 
travelling not because they want to leave. A bank manager who knows the customer personally can make a much better decision. The model is a helpful 
tool but the final call should always come from a human.
""")


Summary

Which model performed better?
Looking at the results, both models did a decent job but in different ways.  Logistic Regression was better at catching customers who actually left the 
bank. SVM was more careful and precise but missed some churners. So it  depends on what matters more to the bank.

Which metric is most important for this business problem?
For a bank trying to stop customers from leaving, Recall is the most  important metric. We would rather flag too many customers than miss someone 
who is about to leave. Losing a customer completely is much more costly than sending an extra retention offer to someone who was going to stay anyway.

What do false positives and false negatives mean?
A false positive is when the model thinks a customer will leave but they actually stay. This means the bank spends money on a retention offer they 
did not need. Not a big deal but a waste of resources. A false negative is when the model thinks a customer will stay but they actually l